In [1]:
from TableTennisEnvironmentV0.TableTennisEnvironment import TableTennisEnv
import pybullet as p
env = TableTennisEnv(show_gui=True)

pybullet build time: Nov 28 2023 23:51:11


In [2]:


def calculate_joint_angles(robot_id, end_effector_pos, end_effector_orientation):
    """
    Calculates the joint angles for a given end-effector position and orientation.

    Args:
        robot_id (int): Unique ID of the robot in the PyBullet simulation.
        end_effector_pos (tuple of float): The target position of the end-effector (x, y, z).
        end_effector_orientation (tuple of float): The target orientation of the end-effector as a quaternion (x, y, z, w).

    Returns:
        list of float: The joint angles required to achieve the given end-effector position and orientation.
    """
    # Index of the end effector link
    end_effector_link_index = 6  # Assuming the last link is the end effector, adjust as needed

    # Calculate the Inverse Kinematics
    joint_angles = p.calculateInverseKinematics(
        bodyUniqueId=robot_id,
        endEffectorLinkIndex=end_effector_link_index,
        targetPosition=end_effector_pos,
        targetOrientation=end_effector_orientation,
        maxNumIterations=20000,
        residualThreshold=0.0001,
        solver = p.IK_DLS,
        # Additional parameters can be adjusted as needed, including joint limits, rest poses, etc.
    )

    return joint_angles


In [7]:
# Desired end-effector position and orientation (example)
end_effector_pos = (1.5, 0.4, 1.75)  # Adjust as needed
end_effector_orientation = p.getQuaternionFromEuler([0, 0, 1])  # No rotation
joint_angles = calculate_joint_angles(env._robotic_arm, end_effector_pos, end_effector_orientation)
for i, angle in enumerate(joint_angles):
    p.setJointMotorControl2(bodyIndex=env._robotic_arm,
                            jointIndex=i,
                            controlMode=p.POSITION_CONTROL,
                            targetPosition=angle,
                           )

In [8]:
# Step simulation to see the result
import time
for i in range(1000):
    p.stepSimulation()
    time.sleep(1./240.)

In [8]:
end_effector_link_index = 7
state = p.getLinkState(env._robotic_arm, end_effector_link_index)

# Position and orientation of the end effector
position = state[0]
orientation = state[1]

print(position)

(1.6741732077763007, -0.02691425070079448, 2.404556360725951)


In [3]:


# Specify the link index you want to highlight
linkIndexToHighlight = 6  # For example, the 7th link

# Change the color of the link to highlight it (e.g., bright red)
p.changeVisualShape(env._robotic_arm, linkIndexToHighlight, rgbaColor=[1, 0, 0, 1])


## Fresh Test

In [3]:
robotId = env._robotic_arm
# Get the number of joints in the robot
numJoints = p.getNumJoints(robotId)

# Initialize a list to store joint positions
jointPositions = []

# Loop through all the joints
for jointIndex in range(numJoints):
    # Get the state of the joint
    jointState = p.getJointState(robotId, jointIndex)
    # The first element in the returned tuple is the position of the joint
    jointPosition = jointState[0]
    # Store the joint position
    jointPositions.append(jointPosition)

# Print all the joint positions
for index, position in enumerate(jointPositions):
    print(f"Joint {index} Position: {position*180.0 / 3.1416}")

Joint 0 Position: 0.0
Joint 1 Position: 0.0
Joint 2 Position: 113.44537815126056
Joint 3 Position: 171.8869385521735
Joint 4 Position: 171.8814255474845
Joint 5 Position: 171.88416299502808
Joint 6 Position: 171.8878205736685
Joint 7 Position: 85.94736974665794
Joint 8 Position: 0.0
Joint 9 Position: 0.0
Joint 10 Position: 57.29571554709308
Joint 11 Position: 57.29267879852177
Joint 12 Position: 57.29606917831401
Joint 13 Position: 57.29761989825154
Joint 14 Position: 57.29953590040656
Joint 15 Position: 3.9961577195298754


In [4]:
for joint_index in range(numJoints):
    joint_info = p.getJointInfo(robotId, joint_index)
    name, joint_type, lower_limit, upper_limit = \
        joint_info[1], joint_info[2], joint_info[8], joint_info[9]
    print(joint_index, name, joint_type, lower_limit, upper_limit)

0 b'connect_root_and_world' 4 0.0 -1.0
1 b'j2s6s300_joint_base' 4 0.0 -1.0
2 b'j2s6s300_joint_1' 0 0.0 -1.0
3 b'j2s6s300_joint_2' 0 0.8203 5.46288
4 b'j2s6s300_joint_3' 0 0.33161 5.95157
5 b'j2s6s300_joint_4' 0 0.0 -1.0
6 b'j2s6s300_joint_5' 0 0.5236 5.75959
7 b'j2s6s300_joint_6' 0 0.0 -1.0
8 b'j2s6s300_joint_end_effector' 4 0.0 -1.0
9 b'joint_dummy1' 4 0.0 -1.0
10 b'j2s6s300_joint_finger_1' 0 0.0 1.51
11 b'j2s6s300_joint_finger_tip_1' 0 0.0 2.0
12 b'j2s6s300_joint_finger_2' 0 0.0 1.51
13 b'j2s6s300_joint_finger_tip_2' 0 0.0 2.0
14 b'j2s6s300_joint_finger_3' 0 0.0 1.51
15 b'j2s6s300_joint_finger_tip_3' 0 0.0 2.0


In [6]:

for i, angle in enumerate(jointPositions):

    if i == 6:
        angle = angle+0.9
    p.setJointMotorControl2(bodyIndex=env._robotic_arm,
                            jointIndex=i,
                            controlMode=p.POSITION_CONTROL,
                            targetPosition=angle,
                           )

# Step simulation to see the result
import time
for i in range(1000):
    p.stepSimulation()
    time.sleep(1./240.)

In [7]:


# Specify the link index you want to highlight
linkIndexToHighlight = 6  # For example, the 7th link

# Change the color of the link to highlight it (e.g., bright red)
p.changeVisualShape(env._robotic_arm, linkIndexToHighlight, rgbaColor=[1, 0, 0, 1])


In [8]:
# Get the number of joints (and thus links, since each joint connects two links)
numJoints = p.getNumJoints(robotId)

# Loop through all joints to get information about the links
for jointIndex in range(numJoints):
    # Get joint info
    jointInfo = p.getJointInfo(robotId, jointIndex)
    jointName = jointInfo[1].decode("utf-8")
    linkName = jointInfo[12].decode("utf-8")
    
    # Get link state (position, orientation)
    linkState = p.getLinkState(robotId, jointIndex)
    linkWorldPosition = linkState[4]  # World position of the URDF link frame
    linkWorldOrientation = linkState[5]  # World orientation of the URDF link frame
    
    # Print link information
    print(f"Joint Index: {jointIndex}, Joint Name: {jointName}, Link Name: {linkName}")
    print(f"\tLink World Position: {linkWorldPosition}")
    print(f"\tLink World Orientation: {linkWorldOrientation}")



Joint Index: 0, Joint Name: connect_root_and_world, Link Name: root
	Link World Position: (1.7000000476837158, 0.0, 1.25)
	Link World Orientation: (0.0, 0.0, 0.0, 1.0)
Joint Index: 1, Joint Name: j2s6s300_joint_base, Link Name: j2s6s300_link_base
	Link World Position: (1.7000000476837158, 0.0, 1.25)
	Link World Orientation: (0.0, 0.0, 0.0, 1.0)
Joint Index: 2, Joint Name: j2s6s300_joint_1, Link Name: j2s6s300_link_1
	Link World Position: (1.7000000476837158, -1.7462298274040222e-10, 1.406749963760376)
	Link World Orientation: (0.836026668548584, 0.5486888289451599, 7.280004297172127e-07, -1.109234176510654e-06)
Joint Index: 3, Joint Name: j2s6s300_joint_2, Link Name: j2s6s300_link_2
	Link World Position: (1.7014678716659546, -0.0006369315087795258, 1.5255000591278076)
	Link World Orientation: (-0.5622321963310242, -0.4288270175457001, 0.3451935648918152, 0.617125391960144)
Joint Index: 4, Joint Name: j2s6s300_joint_3, Link Name: j2s6s300_link_3
	Link World Position: (1.6784483194351196

In [9]:
# Determine the number of joints in the robot
numJoints = p.getNumJoints(robotId)

# Loop through all the joints to get link information
for jointIndex in range(numJoints):
    # Get the state of the link
    linkState = p.getLinkState(robotId, jointIndex)
    
    # The world position and orientation of the link
    linkWorldPosition = linkState[4]  # World position of the URDF link frame
    linkWorldOrientation = linkState[5]  # World orientation of the URDF link frame in quaternion (x, y, z, w)
    p.addUserDebugText(str(jointIndex),  # Text to display
                   linkWorldPosition, 
                   [1, 0, 0],  # Red color
                   textSize=2,  # Text size
                   lifeTime=0)  # Infinite lifetime (until reset or removed)
    # Print the information
    print(f"Link {jointIndex}: Position = {linkWorldPosition}, Orientation = {linkWorldOrientation}")


Link 0: Position = (1.7000000476837158, 0.0, 1.25), Orientation = (0.0, 0.0, 0.0, 1.0)
Link 1: Position = (1.7000000476837158, 0.0, 1.25), Orientation = (0.0, 0.0, 0.0, 1.0)
Link 2: Position = (1.7000000476837158, -1.7462298274040222e-10, 1.406749963760376), Orientation = (0.836026668548584, 0.5486888289451599, 7.280004297172127e-07, -1.109234176510654e-06)
Link 3: Position = (1.7014678716659546, -0.0006369315087795258, 1.5255000591278076), Orientation = (-0.5622321963310242, -0.4288270175457001, 0.3451935648918152, 0.617125391960144)
Link 4: Position = (1.6784483194351196, -0.05372075363993645, 1.931396722793579), Orientation = (0.5912081003189087, 0.3879133462905884, 0.3880515694618225, 0.5911110639572144)
Link 5: Position = (1.6679712533950806, -0.04923092573881149, 2.1386966705322266), Orientation = (0.795081377029419, 0.6065027713775635, 0.00010027371172327548, -6.239936919882894e-05)
Link 6: Position = (1.6679625511169434, -0.04925384372472763, 2.2424466609954834), Orientation = 

In [12]:
# Draw a text marker at the point
point_to_highlight = [1.3, 0.0, 1.8]
p.addUserDebugText('X',  # Text to display
                   point_to_highlight, 
                   [1, 0, 0],  # Red color
                   textSize=2,  # Text size
                   lifeTime=0)  # Infinite lifetime (until reset or removed)


18

In [12]:
for i in range(1000):
    p.stepSimulation()
    time.sleep(1./240.)

In [21]:

def calculate_joint_angles(robot_id, end_effector_pos, end_effector_orientation):
    """
    Calculates the joint angles for a given end-effector position and orientation.

    Args:
        robot_id (int): Unique ID of the robot in the PyBullet simulation.
        end_effector_pos (tuple of float): The target position of the end-effector (x, y, z).
        end_effector_orientation (tuple of float): The target orientation of the end-effector as a quaternion (x, y, z, w).

    Returns:
        list of float: The joint angles required to achieve the given end-effector position and orientation.
    """
    # Index of the end effector link
    end_effector_link_index = 5  # Assuming the last link is the end effector, adjust as needed

    # Calculate the Inverse Kinematics
    joint_angles = p.calculateInverseKinematics(
        bodyUniqueId=robot_id,
        endEffectorLinkIndex=end_effector_link_index,
        targetPosition=end_effector_pos,
        targetOrientation=end_effector_orientation,
        maxNumIterations=20000,
        residualThreshold=0.0001,
        solver = p.IK_DLS,
        # Additional parameters can be adjusted as needed, including joint limits, rest poses, etc.
    )

    return joint_angles

In [22]:
# Desired end-effector position and orientation (example)
end_effector_pos = (1.3, 0, 1.8)  # Adjust as needed
end_effector_orientation = p.getQuaternionFromEuler([0, 0, 0])  # No rotation
joint_angles = calculate_joint_angles(env._robotic_arm, end_effector_pos, end_effector_orientation)
for i, angle in enumerate(joint_angles):
    p.setJointMotorControl2(bodyIndex=env._robotic_arm,
                            jointIndex=i,
                            controlMode=p.POSITION_CONTROL,
                            targetPosition=angle,
                           )

In [6]:
for i in range(200):
    p.stepSimulation()
    time.sleep(1./240.)

NameError: name 'time' is not defined

## Make ROS node to copy the moveit joint velocities in the real robot. 

In [2]:
import rospy
from sensor_msgs.msg import JointState

In [3]:
def joint_state_callback(data):
    joint_names = data.name
    joint_positions = data.position
    print(joint_positions)
    for i, angle in enumerate(joint_positions):
        p.setJointMotorControl2(bodyIndex=env._robotic_arm,
                                jointIndex=i+2,
                                controlMode=p.POSITION_CONTROL,
                                targetPosition=angle,
                               )
    p.stepSimulation()

In [ ]:
def listener():
    rospy.init_node('joint_state_listener', anonymous = True)
    rospy.Subscriber("/move_group/fake_controller_joint_states", JointState, joint_state_callback)
    while True:
        continue


listener()
    

(2.812538792500489e-08, 3.1415000229219068, 3.141500025397594, -1.4470908567694415e-11, 3.1415926560640464, -2.811333504497254e-08)
(2.1342341943121435e-05, 3.1415173937927983, 3.1415192724143837, -1.0980935790215456e-08, 3.1415945311234585, -2.1333195875954796e-05)
(0.02103524230160499, 3.1586435096967587, 3.1604950993839958, -1.0822928695507785e-05, 3.1434431710114863, -0.021026227839205793)
(0.04202721915105218, 3.175751758487654, 3.1794511294978665, -2.1623596705972403e-05, 3.1452898822743616, -0.0420092087672711)
(0.06301324223367936, 3.192855155018816, 3.1984017832822973, -3.2421201419478576e-05, 3.1471360697710224, -0.0629862384799492)
(0.08400125928037586, 3.2099601766105246, 3.217354237642406, -4.321983205560709e-05, 3.148982432681173, -0.08396526130220139)
(0.1292445161293732, 3.2468329732906245, 3.2582094913152755, -6.649812549327675e-05, 3.152962583411684, -0.1291891295636163)
(0.18088918325262607, 3.2889228545916733, 3.3048453023315676, -9.307003475697719e-05, 3.1575058805